# RAG Retrieval — Serve the Prebuilt Vector DB (Milestone 3)

**This notebook builds nothing.** It restores the production index created by
`08c_rag_vector_db_bge_m3.ipynb` and exposes the single LLM-facing retrieval tool,
`search_agri_knowledge(...)`.

**What it needs on Drive** (written by section 8 of the build notebook):

| Artifact | Purpose |
|---|---|
| `agri_knowledge-*.snapshot` (~3.8 GB) | the entire index: 723,439 vectors + payloads + HNSW graph |
| `manifest.json` | the query-side contract: model, prefixes, dim, tiers, fusion weights |

**Why the manifest matters.** Restoring vectors alone is not enough. bge-m3 takes
**no** `query:`/`passage:` prefixes — adding e5-style prefixes here would silently
change every result with no error, and the tier thresholds are on bge-m3's own
cosine scale (~0.4–0.8), not e5's (~0.75–0.92). Every setting below is read from
the manifest rather than retyped, so this notebook cannot drift from the build.

**Run order:** deps -> Qdrant server -> restore -> embedder -> tool -> query.
Roughly 5–10 minutes, most of it the snapshot upload. No GPU strictly required
(CPU works for single queries), but a GPU makes query embedding faster.

In [ ]:
# Query-time deps only — no chunking / preprocessing libraries needed here.
!pip install qdrant-client sentence-transformers transformers sentencepiece --break-system-packages -q

## 1. Qdrant Server

The index was built with HNSW, which only exists in **server mode**.
`QdrantClient(path=...)` (local mode) ignores the HNSW graph and the payload
indexes and brute-forces every query — measured at 3.7 s per query on 107k chunks
versus **30–330 ms** in server mode at 723k. Local mode also cannot load a snapshot.

Colab has no Docker daemon, so we run Qdrant's static binary directly. The **musl**
build is required: the `linux-gnu` asset needs GLIBC 2.38 and Colab ships 2.35.

In [ ]:
import os, subprocess, tarfile, time, requests

QDRANT_STORAGE = "/content/qdrant_serve_storage"
QDRANT_URL     = "http://localhost:6333"
QDRANT_BIN     = "/content/qdrant"
os.makedirs(QDRANT_STORAGE, exist_ok=True)

def qdrant_alive(url=QDRANT_URL, timeout=1):
    try:
        return requests.get(f"{url}/readyz", timeout=timeout).ok
    except Exception:
        return False

def binary_ok(path=QDRANT_BIN):
    """Existence is not enough — a glibc-linked build downloads fine and only
    fails at exec, so actually run it."""
    if not os.path.exists(path):
        return False
    try:
        return subprocess.run([path, "--version"], capture_output=True,
                              timeout=60).returncode == 0
    except Exception:
        return False

if qdrant_alive():
    print("Qdrant already running on :6333")
else:
    if not binary_ok():
        if os.path.exists(QDRANT_BIN):
            os.remove(QDRANT_BIN)
        rel = requests.get("https://api.github.com/repos/qdrant/qdrant/releases/latest",
                           timeout=30).json()
        asset = None
        for suffix in ("x86_64-unknown-linux-musl.tar.gz",     # static — no libc dependency
                       "x86_64-unknown-linux-gnu.tar.gz"):     # needs GLIBC 2.38; last resort
            asset = next((a for a in rel["assets"] if a["name"].endswith(suffix)), None)
            if asset:
                break
        if asset is None:
            raise RuntimeError("no linux x86_64 asset in " + rel["tag_name"])
        print(f"downloading qdrant {rel['tag_name']} ({asset['name']}) ...")
        with open("/content/q.tar.gz", "wb") as f:
            f.write(requests.get(asset["browser_download_url"], timeout=600).content)
        with tarfile.open("/content/q.tar.gz") as t:
            try:
                t.extractall("/content", filter="data")
            except TypeError:
                t.extractall("/content")
        os.chmod(QDRANT_BIN, 0o755)
        if not binary_ok():
            raise RuntimeError("downloaded qdrant will not execute here")

    env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=QDRANT_STORAGE,
                           QDRANT__TELEMETRY_DISABLED="true")
    qdrant_proc = subprocess.Popen([QDRANT_BIN], env=env, cwd="/content",
                                   stdout=open("/content/qdrant.log", "w"),
                                   stderr=subprocess.STDOUT)
    for _ in range(90):
        if qdrant_alive():
            break
        if qdrant_proc.poll() is not None:
            raise RuntimeError("qdrant exited early — /content/qdrant.log:\n"
                               + open("/content/qdrant.log").read()[-1500:])
        time.sleep(1)
    else:
        raise RuntimeError("qdrant not ready in 90s — see /content/qdrant.log")
    print("Qdrant server ready on :6333")


## 2. Restore the Index

Uploads the snapshot into the running server and reads the manifest. The upload
is the slow part (~3.8 GB from Drive); everything after it is fast.

If the collection is already present — e.g. you re-ran this cell in the same
session — the restore is skipped rather than repeated.

In [ ]:
import json, os, time, requests

# --- point this at the folder written by section 8 of the build notebook -----
ARTIFACT_DIR = "/content/drive/MyDrive/rag_production_bge_m3"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"(not on Colab / Drive already available: {type(e).__name__})")

assert os.path.isdir(ARTIFACT_DIR), (
    f"artifact folder not found: {ARTIFACT_DIR}\n"
    "Run section 8 (EXPORT) of 08c_rag_vector_db_bge_m3.ipynb first.")

MANIFEST = json.load(open(os.path.join(ARTIFACT_DIR, "manifest.json"), encoding="utf-8"))
print("manifest:")
for k in ("embed_model", "embed_dim", "query_prefix", "doc_prefix", "collection",
          "n_chunks", "tiers", "built_utc"):
    print(f"   {k:<14} {MANIFEST.get(k)}")

COLLECTION_NAME = MANIFEST["collection"]

_existing = requests.get(f"{QDRANT_URL}/collections", timeout=30).json()["result"]["collections"]
if any(c["name"] == COLLECTION_NAME for c in _existing):
    print(f"\ncollection '{COLLECTION_NAME}' already present — skipping restore")
else:
    snap = os.path.join(ARTIFACT_DIR, MANIFEST["snapshot"])
    assert os.path.exists(snap), f"snapshot missing: {snap}"
    gb = os.path.getsize(snap) / 1e9
    print(f"\nuploading snapshot ({gb:.2f} GB) — the slow step, several minutes ...")
    t0 = time.time()
    with open(snap, "rb") as fh:
        r = requests.post(
            f"{QDRANT_URL}/collections/{COLLECTION_NAME}/snapshots/upload?priority=snapshot",
            files={"snapshot": (MANIFEST["snapshot"], fh)}, timeout=7200)
    r.raise_for_status()
    print(f"restored in {(time.time()-t0)/60:.1f} min")

from qdrant_client import QdrantClient
qdrant_client = QdrantClient(url=QDRANT_URL, timeout=600)
info = qdrant_client.get_collection(COLLECTION_NAME)
print(f"\ncollection '{COLLECTION_NAME}'")
print(f"   points        : {info.points_count:,}")
print(f"   indexed vecs  : {info.indexed_vectors_count:,}")
print(f"   status        : {info.status}")

# A restored-but-unindexed collection still answers queries, just slowly — worth
# knowing about rather than silently blaming the model for bad latency.
if info.indexed_vectors_count < info.points_count:
    print("\n[WARN] HNSW not fully built yet; queries work but are slower until it is.")
if MANIFEST.get("n_chunks") and info.points_count != MANIFEST["n_chunks"]:
    print(f"\n[WARN] point count {info.points_count:,} != manifest {MANIFEST['n_chunks']:,}")


## 3. Embedder and Query Contract

Everything here comes from the manifest. The three settings that silently corrupt
results if they disagree with the build:

- **model** — a different checkpoint produces vectors in a different space
- **prefixes** — bge-m3 uses none; e5-style `query: ` would shift every vector
- **max_seq_length** — pinned to the build value so long queries truncate identically

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = MANIFEST["embed_model"]
MODEL_MAX_TOKENS = MANIFEST["max_seq_length"]
QUERY_PREFIX     = MANIFEST["query_prefix"]
DOC_PREFIX       = MANIFEST["doc_prefix"]

embed_doc   = lambda t: DOC_PREFIX + t
embed_query = lambda t: QUERY_PREFIX + t

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"loading {EMBED_MODEL_NAME} on {device} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
embed_model.max_seq_length = MODEL_MAX_TOKENS

EMBED_DIM = embed_model.get_sentence_embedding_dimension()
assert EMBED_DIM == MANIFEST["embed_dim"], (
    f"dim mismatch: model gives {EMBED_DIM}, index was built with "
    f"{MANIFEST['embed_dim']} — wrong model for this snapshot")
print(f"   dim {EMBED_DIM}  max_seq {MODEL_MAX_TOKENS}  "
      f"prefixes query={QUERY_PREFIX!r} doc={DOC_PREFIX!r}")

# --- retrieval config, also from the manifest -------------------------------
# Tiers are on THIS model's cosine scale. bge-m3 puts unrelated documents at
# ~0.40 and good matches at ~0.60-0.80; the e5-era 0.837/0.857 pair would abstain
# on literally every query here. Measured by check D2 on the full 723k index.
TOP_K_DEFAULT  = MANIFEST["top_k_default"]
TIER_GROUNDED  = MANIFEST["tiers"]["grounded"]     # >= : cite-and-answer
TIER_FALLBACK  = MANIFEST["tiers"]["fallback"]     # >= : answer + "verify with KVK"
FUSION_WEIGHTS = MANIFEST["fusion_weights"]        # <  : abstain / out-of-scope

print(f"   tiers: abstain < {TIER_FALLBACK} <= fallback < {TIER_GROUNDED} <= grounded")
print(f"   fusion: {FUSION_WEIGHTS}")


## 4. Filter Canonicalization

The `crop` and `district` filter values must be normalised **exactly** as they
were at ingestion, or a filter silently matches nothing. This is the same code the
build notebook used.

The failure it prevents: KCC stores crops as `Paddy (Dhan)`, `Maize (Makka)`.
An earlier version only matched bare names, so `crop="rice"` matched **zero** of
112,269 rice chunks — no error, just empty results.

In [ ]:
import re

# District renames/typos -> canonical post-bifurcation names (extends the
# VALIDATED_ALIASES map from 04_pdfs_rag_eda.ipynb; KCC adds raw-source typos).
DISTRICT_CANON = {
    "allahabad": "prayagraj", "faizabad": "ayodhya",
    "prabuddh nagar": "shamli", "prabudh nagar": "shamli",
    "bhim nagar": "sambhal", "panchsheel nagar": "hapur",
    "jyotiba phule nagar": "amroha", "jyotibaphule nagar": "amroha",
    "kanshi ram nagar": "kasganj", "kanshiram nagar": "kasganj",
    "chhatrapati shahuji maharaj nagar": "amethi",
    "mahamaya nagar": "hathras", "ramabai nagar": "kanpur dehat",
    "banaras": "varanasi", "kashi": "varanasi",
    # raw KCC source typos observed in the corpus
    "kanpur city": "kanpur nagar", "maharahganj": "maharajganj",
    "sant ravidas nagar": "bhadohi",
}

def canon_district(raw):
    """Lowercase, collapse spaces, apply the shared rename/typo map."""
    if not raw or str(raw).lower() in ("unknown", "nan", "none", ""):
        return None
    d = re.sub(r"\s+", " ", str(raw).strip().lower())
    return DISTRICT_CANON.get(d, d)

# Canonical crop vocabulary. Keys are ANY surface form (English name, Hindi /
# Hinglish vernacular, or an alias appearing inside the KCC parentheses); values
# are the single canonical token stored in the payload AND accepted by the tool's
# `crop` filter. Every canonical value is also a key, so canon_crop() is
# idempotent -- payload value and user filter term must converge on one string.
CROP_CANON = {
    "rice": "rice", "paddy": "rice", "dhan": "rice", "chawal": "rice",
    "wheat": "wheat", "gehun": "wheat", "gehu": "wheat", "kanak": "wheat",
    "maize": "maize", "makka": "maize", "makai": "maize", "bhutta": "maize", "corn": "maize",
    "मक्के": "maize",   # oblique Devanagari form -- makke, e.g. "makke mein"
    "sugarcane": "sugarcane", "ganna": "sugarcane", "ganne": "sugarcane", "noble cane": "sugarcane",
    "गन्ने": "sugarcane",   # oblique Devanagari form -- ganne, e.g. "ganne mein"
    "potato": "potato", "aloo": "potato", "alu": "potato",
    "mustard": "mustard", "sarson": "mustard", "raya": "mustard",
    "indian mustard": "mustard", "indian rapeseed and mustard": "mustard", "yellow sarson": "mustard",
    "urad": "urad", "black gram": "urad", "urd": "urad", "urd bean": "urad",
    "gram": "gram", "bengal gram": "gram", "chana": "gram", "chane": "gram", "chick pea": "gram", "kabuli": "gram",
    "moong": "moong", "green gram": "moong", "moong bean": "moong", "mung": "moong",
    "arhar": "arhar", "pigeon pea": "arhar", "red gram": "arhar", "tur": "arhar",
    "masur": "masur", "lentil": "masur",
    "okra": "okra", "bhindi": "okra", "ladysfinger": "okra",
    "bajra": "bajra", "pearl millet": "bajra", "bulrush millet": "bajra", "spiked millet": "bajra",
    "jowar": "jowar", "sorghum": "jowar", "great millet": "jowar",
    "barley": "barley", "jau": "barley",
    "sesame": "sesame", "til": "sesame", "gingelly": "sesame", "sesamum": "sesame",
    "groundnut": "groundnut", "pea nut": "groundnut", "peanut": "groundnut", "mung phalli": "groundnut",
    "colocasia": "arvi", "arvi": "arvi", "arbi": "arvi", "arum": "arvi",
    "cotton": "cotton", "kapas": "cotton",
    "soybean": "soybean", "bhat": "soybean",
    "linseed": "linseed", "alsi": "linseed",
    "spinach": "spinach", "palak": "spinach",
    "methi": "fenugreek", "fenugreek": "fenugreek",
    "rajma": "rajma", "french bean": "rajma",
    "sunflower": "sunflower", "suryamukhi": "sunflower",
    "finger millet": "ragi", "fingermillet": "ragi", "ragi": "ragi", "mandika": "ragi",
    "pea": "pea", "peas": "pea", "matar": "pea", "field peas": "pea", "garden peas": "pea",
}

_CROP_PAREN = re.compile(r"^([^(]+?)\s*\((.*)\)\s*$")
_CROP_NULLS = ("unknown", "nan", "none", "", "na", "n/a", "other", "others")

def canon_crop(raw):
    """Normalise ANY crop surface form to one canonical token.

    KCC stores crops as 'Canonical (vernacular/alias/alias)' -- 'Paddy (Dhan)',
    'Maize (Makka)', 'Bengal Gram (Gram/Chick Pea/Chana)'. The previous version
    only matched bare names, so the parenthesised form never hit CROP_CANON:
    crop='rice' resolved to 'rice' while 112,266 chunks sat under the payload
    value 'paddy (dhan)'. Filtering rice or maize matched ZERO chunks -- silently,
    with no error (this is what eval check A2 was actually reporting).

    Order: whole string -> base before '(' -> each alias inside '()' -> the base.
    Applied identically to payload values at ingestion and to the caller's filter
    term at query time, so both sides always land on the same token.
    """
    if raw is None:
        return None
    c = re.sub(r"\s+", " ", str(raw).strip().lower())
    if c in _CROP_NULLS:
        return None
    if c in CROP_CANON:                          # 'rice', 'dhan', 'paddy'
        return CROP_CANON[c]
    m = _CROP_PAREN.match(c)
    if m:
        base, inner = m.group(1).strip(), m.group(2)
        if base in CROP_CANON:                   # 'paddy (dhan)' -> 'paddy' -> 'rice'
            return CROP_CANON[base]
        for alias in re.split(r"[/,]", inner):   # 'bhindi(okra/ladysfinger)'
            alias = alias.strip()
            if alias in CROP_CANON:
                return CROP_CANON[alias]
        return base                              # unmapped: keep the English base
    return c



## 5. The LLM Tool — `search_agri_knowledge`

JSON in, JSON out, **never raises** (errors come back as `tier: "error"`), every
hit cited. This is the single retrieval entrypoint the agent layer calls.

**Weighted per-source fusion.** KCC outnumbers the PDF corpus ~100:1, so a flat
top-k would drown scheme content. The tool runs one filtered sub-query per corpus
and fuses with intent weights: `policy` → PDF-heavy, `field_practice` → KCC-heavy.

**Tiers are decided on the RAW cosine, never the fused score** — fusion multiplies
by up to 2.0, which would otherwise manufacture confidence out of a weighting choice.

In [ ]:
# Filter primitives for the tool (self-contained -- the build cell no longer
# exports these into the global scope).
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

RAG_TOOL_SPEC = {
    "name": "search_agri_knowledge",
    "description": (
        "Search the UP agricultural knowledge base: government scheme guidelines, "
        "pest/disease advisories and district contingency plans (PDF corpus) plus "
        "Kisan Call Centre farmer Q&A with expert answers (KCC corpus). Returns "
        "cited chunks with a relevance tier. Query may be English or Hindi."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query":       {"type": "string", "description": "The farmer's question, English or Hindi"},
            "top_k":       {"type": "integer", "default": 5, "minimum": 1, "maximum": 20},
            "intent":      {"type": "string", "enum": ["policy", "field_practice", "general"],
                            "description": "Weights the PDF-vs-KCC fusion; default general"},
            "source_type": {"type": "string", "enum": ["pdf", "kcc"],
                            "description": "Pin one corpus; omit for weighted search over both"},
            "doc_category": {"type": "string", "enum": ["scheme_eligibility", "crop_advisory",
                                                        "contingency_plan", "policy_guideline"],
                             "description": "PDF corpus only"},
            "query_type":  {"type": "string", "description": "KCC only, e.g. 'Plant Protection'"},
            "crop":        {"type": "string", "description": "Canonical crop name (rice, wheat, ...)"},
            "district":    {"type": "string", "description": "Canonical UP district name"},
            "season":      {"type": "string", "enum": ["Rabi", "Kharif", "Zaid"], "description": "KCC only"},
            "language":    {"type": "string", "enum": ["en", "hi", "mixed"]},
            "year_from":   {"type": "integer", "description": "Only content from this year onward"},
            "only_tables": {"type": "boolean", "description": "PDF dosage/scheme tables only"},
        },
        "required": ["query"],
    },
}


def _citation(p):
    if p.get("source_type") == "pdf":
        # `district` is populated for ACP contingency plans (named after the
        # district). It was missing from this branch, so a district-filtered
        # search returned correct PDF hits whose citation reported district=None
        # -- eval check A4 then flagged them as violations. Retrieval was fine;
        # the citation was lossy.
        return {"corpus": "pdf", "file": p.get("filename"),
                "pages": [p.get("page_start"), p.get("page_end")],
                "section": p.get("heading_hierarchy") or None,
                "doc_category": p.get("doc_category"),
                "district": p.get("district"), "year": p.get("year")}
    return {"corpus": "kcc", "record": "KCC Q&A", "crop": p.get("crop"),
            "district": p.get("district"), "season": p.get("season"),
            "query_type": p.get("query_type"), "year": p.get("year")}


def search_agri_knowledge(query, top_k=TOP_K_DEFAULT, intent="general", source_type=None,
                          doc_category=None, query_type=None, crop=None, district=None,
                          season=None, language=None, year_from=None, only_tables=None):
    """LLM tool entrypoint: JSON-in/JSON-out, never raises, always cites."""
    weights = FUSION_WEIGHTS.get(intent, FUSION_WEIGHTS["general"])

    def sub_search(stype):
        must = [FieldCondition(key="source_type", match=MatchValue(value=stype))]
        if doc_category: must.append(FieldCondition(key="doc_category", match=MatchValue(value=doc_category)))
        if query_type:   must.append(FieldCondition(key="query_type", match=MatchValue(value=query_type)))
        if crop:         must.append(FieldCondition(key="crop", match=MatchValue(value=canon_crop(crop))))
        if district:     must.append(FieldCondition(key="district", match=MatchValue(value=canon_district(district))))
        if season:       must.append(FieldCondition(key="season", match=MatchValue(value=season)))
        if language:     must.append(FieldCondition(key="language", match=MatchValue(value=language)))
        if year_from:    must.append(FieldCondition(key="year", range=Range(gte=year_from)))
        if only_tables:  must.append(FieldCondition(key="has_table", match=MatchValue(value=True)))
        return qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=qvec,
            query_filter=Filter(must=must),
            limit=top_k,
            with_payload=True,
        ).points

    try:
        # e5 is asymmetric: the QUERY side takes "query: ", never "passage: ".
        qvec = embed_model.encode(embed_query(query), normalize_embeddings=True).tolist()
        sources = [source_type] if source_type else ["pdf", "kcc"]
        hits = []
        for stype in sources:
            for h in sub_search(stype):
                hits.append({
                    "raw_score": round(float(h.score), 4),
                    "fused_score": round(float(h.score) * weights.get(stype, 1.0), 4),
                    "text": h.payload.get("text", ""),
                    "source_type": stype,
                    "has_table": bool(h.payload.get("has_table", False)),
                    "chunk_id": h.payload.get("chunk_id"),
                    "citation": _citation(h.payload),
                })
        hits.sort(key=lambda x: x["fused_score"], reverse=True)
        hits = hits[:top_k]
    except Exception as e:
        return {"query": query, "tier": "error", "top_score": 0.0,
                "results": [], "error": str(e)}

    best_raw = max((h["raw_score"] for h in hits), default=0.0)  # tier on RAW cosine
    tier = ("grounded" if best_raw >= TIER_GROUNDED
            else "fallback_with_disclaimer" if best_raw >= TIER_FALLBACK
            else "abstain_out_of_scope")
    return {"query": query, "intent": intent, "tier": tier,
            "top_score": round(best_raw, 4), "results": hits}



print("Tool ready: search_agri_knowledge(...)")
_s = search_agri_knowledge("interest subvention on crop loans", top_k=3, intent="policy")
print(f"smoke: tier={_s['tier']} top={_s['top_score']} "
      f"sources={[r['source_type'] for r in _s['results']]}")
assert _s["tier"] != "error", _s.get("error")


## 6. Worked Examples

A spread across the query types the system must handle, with the retrieved text
shown so you can judge quality directly rather than trusting a score.

In [ ]:
import time

def ask(query, show=3, **kw):
    """Query and pretty-print. Same call the agent layer makes."""
    t0 = time.time()
    out = search_agri_knowledge(query, **kw)
    ms = (time.time() - t0) * 1000
    params = ", ".join(f"{k}={v!r}" for k, v in kw.items()) or "—"
    print("=" * 96)
    print(f"Q      : {query}")
    print(f"params : {params}")
    if out["tier"] == "error":
        print(f"ERROR  : {out['error'][:200]}")
        return out
    print(f"result : tier={out['tier']}  top={out['top_score']:.3f}  {ms:.0f}ms  "
          f"({sum(r['source_type']=='pdf' for r in out['results'])} pdf / "
          f"{sum(r['source_type']=='kcc' for r in out['results'])} kcc)")
    for i, h in enumerate(out["results"][:show], 1):
        c = h["citation"]
        if c["corpus"] == "pdf":
            pages = c.get("pages") or [None, None]
            where = f"{c.get('file')} p.{pages[0]}"
        else:
            where = f"KCC | {c.get('crop')} | {c.get('district')} | {c.get('year')}"
        txt = re.sub(r"\s+", " ", h["text"]).strip()
        print(f"\n  {i}. {h['raw_score']:.3f}  <{h['source_type']}>  {where}")
        print(f"     {txt[:300]}{' ...' if len(txt) > 300 else ''}")
    return out

ask("who is eligible for interest subvention on crop loans", intent="policy")
ask("how much urea should be applied in wheat at tillering stage", intent="field_practice")
ask("टमाटर के पौधे में पत्तियां मुड़ रही हैं क्या करें")
ask("dhan ki nursery me pili patti ho rahi hai kya karein", intent="field_practice")
ask("approved fungicides and dosage for rice blast", only_tables=True)
ask("fertilizer dose for paddy nursery", source_type="kcc", crop="rice")
ask("how do I repair my motorcycle engine")          # expect abstain


## 7. Ask Anything

Edit `MY_QUERY` and re-run. Optional filters are listed in `RAG_TOOL_SPEC`:
`intent`, `source_type`, `crop`, `district`, `season`, `query_type`,
`doc_category`, `language`, `year_from`, `only_tables`, `top_k`.

In [ ]:
MY_QUERY  = "गेहूं में खरपतवार नियंत्रण के लिए कौन सी दवा डालें"
MY_PARAMS = dict(intent="field_practice", top_k=5)

_ = ask(MY_QUERY, show=5, **MY_PARAMS)

# Raw JSON — exactly what the LLM layer receives.
# print(json.dumps(_, indent=2, ensure_ascii=False)[:2000])


## 8. Health Check

A short regression suite: filters must be honoured, routing must respect the
fusion weights, and off-domain queries must score clearly below in-domain ones or
tier-based abstention cannot work. Run this after any restore to confirm the index
came back intact.

In [ ]:
report, timings = [], []

def _t(**kw):
    t0 = time.time(); out = search_agri_knowledge(**kw)
    timings.append((time.time() - t0) * 1000); return out

def chk(label, out, pred):
    if out["tier"] == "error":
        report.append((label, "FAIL", out.get("error", "")[:70]))
    elif not out["results"]:
        report.append((label, "WARN", "0 results"))
    else:
        bad = [h for h in out["results"] if not pred(h)]
        report.append((label, "PASS" if not bad else "FAIL",
                       f"{len(out['results'])} hits, {len(bad)} violate, top={out['top_score']:.3f}"))

chk("filter: pdf + scheme_eligibility",
    _t(query="eligibility for interest subvention", source_type="pdf",
       doc_category="scheme_eligibility"),
    lambda h: h["source_type"] == "pdf" and h["citation"]["doc_category"] == "scheme_eligibility")
chk("filter: kcc + crop=rice  (canon: 'Paddy (Dhan)')",
    _t(query="fertilizer dose for paddy nursery", source_type="kcc", crop="rice"),
    lambda h: h["source_type"] == "kcc" and h["citation"]["crop"] == "rice")
chk("filter: only_tables",
    _t(query="approved fungicides and dosage for rice blast", only_tables=True),
    lambda h: h["has_table"])
chk("filter: district canon allahabad->prayagraj",
    _t(query="contingency plan for delayed monsoon", district="allahabad"),
    lambda h: h["citation"].get("district") == "prayagraj")
chk("filter: season=Rabi",
    _t(query="wheat sowing time", season="Rabi"),
    lambda h: h["citation"].get("season") == "Rabi")

pol = _t(query="pm kisan samman nidhi eligibility and benefits", intent="policy")
report.append(("routing: policy -> pdf-heavy",
               "PASS" if sum(r["source_type"] == "pdf" for r in pol["results"][:3]) >= 2 else "WARN",
               f"{sum(r['source_type']=='pdf' for r in pol['results'][:3])}/3 pdf"))
fld = _t(query="yellowing in onion nursery what spray to use", intent="field_practice")
report.append(("routing: field -> kcc-heavy",
               "PASS" if sum(r["source_type"] == "kcc" for r in fld["results"][:3]) >= 2 else "WARN",
               f"{sum(r['source_type']=='kcc' for r in fld['results'][:3])}/3 kcc"))

in_s  = [_t(query=q, top_k=1)["top_score"] for q in
         ["recommended fertilizer schedule for wheat in up", "fall army worm control in maize",
          "गेहूं में पीला रतुआ की रोकथाम", "pm kisan samman nidhi eligibility"]]
off_s = [_t(query=q, top_k=1)["top_score"] for q in
         ["how do I repair my car engine", "best chess opening strategy",
          "शेयर बाजार में निवेश कैसे करें"]]
margin = min(in_s) - max(off_s)
report.append(("abstention: domain separation", "PASS" if margin > 0.02 else "WARN",
               f"in-min={min(in_s):.3f} off-max={max(off_s):.3f} margin={margin:+.3f}"))

import numpy as np
print("=" * 84)
print("RETRIEVAL HEALTH CHECK")
print("=" * 84)
for lab, st, det in report:
    print(f"  [{st:4}] {lab:<44} {det}")
print("-" * 84)
n_fail = sum(1 for _, s, _ in report if s == "FAIL")
print(f"  {sum(1 for _,s,_ in report if s=='PASS')} pass / {n_fail} fail / "
      f"{sum(1 for _,s,_ in report if s=='WARN')} warn"
      f"  |  latency p50={np.percentile(timings,50):.0f}ms p95={np.percentile(timings,95):.0f}ms")
if n_fail:
    print("\n  FAILURES mean the restored index does not match the manifest —")
    print("  re-check ARTIFACT_DIR and that the snapshot finished uploading.")
